In [ ]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.append("../src")

In [ ]:
import logging
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import relplot as rp

from benchmark import (
    INVALID_ANSWER,
    NO_ANSWER,
    VALID_ANSWER,
    aggregate_responses,
    brier_score_decomposition,
    calibration_error,
    detect_names_from_dict,
    empirical_distr,
    extract_predictions,
    kl_div,
    load_predictions,
    load_responses_all,
    plot_annotation,
    plot_calibration_curve,
    plot_heatmap,
    sample_predictions,
    save_predictions,
)
from utils_ext.plot import Plotter
from utils_ext.tools import setup_logging

plt.ioff()
setup_logging()

logger = logging.getLogger(__name__)

PATH_OUTPUT = "../results"

# setup plotter

FONTSIZE_DEFAULT = 6
FONTSIZE_SMALL = 5

Plotter.setup()
Plotter.configure(
    basewidth=5.5,
    fontsize=FONTSIZE_DEFAULT,
    latex=False,
    rcparams={
        "lines.linewidth": 1, # default: 1.5
        "axes.labelpad": 1, # default: 4
    },
    save_dir=f"{PATH_OUTPUT}/plots_paper",
    save_format="pdf",
)
Plotter.configure(
    latex=True,
    latex_preamble="\n".join([
        r"\usepackage[utf8]{inputenc}",
        r"\usepackage[T1]{fontenc}",
        r"\usepackage{microtype}",
        r"\usepackage{amsmath,amssymb,amsfonts,mathrsfs}",
        r"\renewcommand{\rmdefault}{ptm}",
        r"\renewcommand{\sfdefault}{phv}",
    ]),
)

Plotter.display_css(".cell-output-ipywidget-background { background: lightgray !important; }")

In [ ]:
DATASET_CACHE = {}
NUM_SEEDS = 10

## Load results

In [ ]:
# OPTION 1: load responses and extract predictions
responses_all = load_responses_all(f"{PATH_OUTPUT}/responses", dataset_cache=DATASET_CACHE)

y_true_all_, y_pred_all_ = extract_predictions(responses_all)
# save_predictions(f"{PATH_OUTPUT}/predictions", y_true_all_, y_pred_all_)

y_true_all, y_pred_all = {}, {}
for seed in range(NUM_SEEDS):
    y_true_all[seed], y_pred_all[seed] = sample_predictions(y_true_all_, y_pred_all_, k=1000, seed=seed)
    # save_predictions(f"{PATH_OUTPUT}/predictions_sampled{seed}", y_true_all[seed], y_pred_all[seed])

In [ ]:
# OPTION 2: load predictions
y_true_all, y_pred_all = {}, {}
for seed in range(NUM_SEEDS):
    y_true_all[seed], y_pred_all[seed] = load_predictions(f"{PATH_OUTPUT}/predictions_sampled{seed}")

## Paper Plots

In [ ]:
COLOR_ACCURACY = "tab:blue"
COLOR_CONFIDENCE = "tab:green"
COLOR_ECE = "tab:red"
COLOR_SMECE = "tab:pink"
COLOR_BRIER = "tab:purple"
COLOR_CONF_N_DISTINCT = "darkorange"
COLOR_CONF_VARIANCE = "orange"
COLOR_KL_DIV = "mediumpurple"

YLIM_CALIBRATION = (0, 1.12)
YLIM_OTHERS_1 = (0, 112)
YLIM_OTHERS_2 = (0, 0.23)

# plotting utils

MODELS_TINY = [
    # "gemma1.1-2b-it",
    "gemma1.1-7b-it",
    "llama3-8b-instruct",
    "qwen1.5-7b-chat",
]
MODELS_LARGE = [
    "llama3-70b-instruct",
    "qwen1.5-32b-chat",
    "qwen1.5-72b-chat",
    "qwen1.5-110b-chat",
    "gpt3.5-turbo",
    "gpt4o-mini",
    "gpt4o",
]

METHODS_SCORE_RANGE = [
    "basic_1s",
    None,
    "basic_1s_scorefloat",
    None,
    "basic_1s_scoreletter",
    None,
    "basic_1s_scoretext",
]
METHODS_SCORE_FORMULATION = [
    "basic_1s",
    "basic_1s_probscore",
    None,
    "advanced_1s",
    "advanced_1s_probscore",
    None,
    "tian2023just_1s_top1_v3",
    "tian2023just_1s_top1_v2",
    "tian2023just_1s_top1",
]
METHODS_ADVANCED_FORMULATION = [
    "basic_1s",
    "advanced_1s",
    None,
    "basic_1s_probscore",
    "advanced_1s_probscore",
]
METHODS_FEW_SHOT = [
    "basic_1s",
    "basic_1s_1shot",
    "basic_1s_5shot",
]
METHODS_OTHERS = [
    "tian2023just_1s_top1",
    "tian2023just_1s_top1_v1",
    None,
    "tian2023just_1s_top1",
    "tian2023just_1s_top4",
    None,
    "xiong2023can_vanilla",
    "xiong2023can_cot",
]
METHODS_COMBO = [
    "basic_1s",
    None,
    "basic_1s_probscore",
    None,
    "advanced_1s",
    None,
    "basic_1s_5shot",
    None,
    "combo_1s_v2",
]

DATASET_NAME_ALIASES = {
    "arc-c":           "arc-c",
    "arc-e":           "arc-e",
    "commonsense_qa":  "commonsense_qa",
    "imdb":            "imdb",
    "logi_qa":         "logi_qa",
    "mmlu":            "mmlu",
    "sciq":            "sciq",
    "social_i_qa":     "social_i_qa",
    "trivia_qa":       "trivia_qa",
    "truthful_qa-mc1": "truthful_qa-mc1",
    "truthful_qa-mc2": "truthful_qa-mc2",
}
MODEL_NAME_ALIASES = {
    "gemma1.1-2b-it":      "gemma1.1-2b",
    "gemma1.1-7b-it":      "gemma1.1-7b",
    "llama3-8b-instruct":  "llama3-8b",
    "llama3-70b-instruct": "llama3-70b",
    "qwen1.5-7b-chat":     "qwen1.5-7b",
    "qwen1.5-32b-chat":    "qwen1.5-32b",
    "qwen1.5-72b-chat":    "qwen1.5-72b",
    "qwen1.5-110b-chat":   "qwen1.5-110b",
    "gpt3.5-turbo":        "gpt3.5-turbo",
    "gpt4o-mini":          "gpt4o-mini",
    "gpt4o":               "gpt4o",
}
METHOD_NAME_ALIASES = {
    "basic_1s":                  "basic",
    "basic_1s_scorefloat":       "basic_scorefloat",
    "basic_1s_scoreletter":      "basic_scoreletter",
    "basic_1s_scoretext":        "basic_scoretext",
    "basic_1s_probscore":        "basic_probscore",
    "basic_1s_1shot":            "basic_1shot",
    "basic_1s_5shot":            "basic_5shot",
    "advanced_1s":               "advanced",
    "advanced_1s_probscore":     "advanced_probscore",
    "combo_1s_v2":               "combo",
    "tian2023just_1s_top1":      "tian2023_top1",
    "tian2023just_1s_top1_v1":   "tian2023_top1_v1",
    "tian2023just_1s_top1_v2":   "tian2023_top1_v2",
    "tian2023just_1s_top1_v3":   "tian2023_top1_v3",
    "tian2023just_1s_top4":      "tian2023_top4",
    "xiong2023can_vanilla":      "xiong2023_vanilla",
    "xiong2023can_cot":          "xiong2023_cot",
}

def translate(names, aliases):
    if isinstance(names, str):
        return aliases[names]
    else:
        return [aliases[name] for name in names]
def translate_dataset(names):
    return translate(names, DATASET_NAME_ALIASES)
def translate_model(names):
    return translate(names, MODEL_NAME_ALIASES)
def translate_method(names):
    return translate(names, METHOD_NAME_ALIASES)

def extract_none_indices(names):
    none_indices = [i for i, name in enumerate(names) if name is None]
    none_indices = list(none_indices - np.arange(len(none_indices)))
    names = [name for name in names if name is not None]
    return names, none_indices

def plot_grouped_bar(axes_scores_args, width=None, alpha=None, with_labels=False, with_lines=False, with_errorbars=False, none_indices=[]):
    def filter_singles(l):
        l_new = []
        for i in range(len(l)):
            if not np.isnan(l[i]):
                left_is_val = i-1 >= 0 and not np.isnan(l[i-1])
                right_is_val = i+1 < len(l) and not np.isnan(l[i+1])
                if left_is_val:
                    l_new.append(l[i])
                elif right_is_val:
                    if len(l_new) > 0:
                        l_new.append(np.nan)
                    l_new.append(l[i])
        return np.asarray(l_new)

    n_bars = len(axes_scores_args)
    n_scores = len(axes_scores_args[0][1])
    if width is None:
        width = 1 / (n_bars + 1.5)
    if alpha is None:
        alpha = 0.5 if with_lines else 1

    x = np.arange(n_scores, dtype=float)
    offset = (n_bars - 1) / 2
    # plot bars
    for i, (ax, scores, args) in enumerate(axes_scores_args):
        mean = np.mean(scores, axis=1)
        if with_errorbars:
            std = np.std(scores, axis=1)
            lower = mean - 2 * std
            upper = mean + 2 * std
            yerr = [mean - lower, upper - mean]
        else:
            yerr = None
        fmt = args.pop("fmt", "{:.2f}")
        args.setdefault("alpha", alpha)
        bar_container = ax.bar(x+(i-offset)*width, mean, width, yerr=yerr, **args)
        if with_labels:
            ax.bar_label(bar_container, fmt=fmt, fontsize=FONTSIZE_SMALL, padding=3, rotation=90)
    # plot lines
    if with_lines:
        x_ = filter_singles(np.insert(x, none_indices, np.nan))
        for i, (ax, scores, args) in enumerate(axes_scores_args):
            mean = np.mean(scores, axis=1)
            ax.plot(x_+(i-offset)*width, filter_singles(np.insert(mean, none_indices, np.nan)), color=args["color"], marker="o", markersize=2)
    # plot errorbars (again to appear on top of lines)
    if with_errorbars:
        for i, (ax, scores, args) in enumerate(axes_scores_args):
            mean = np.mean(scores, axis=1)
            std = np.std(scores, axis=1)
            lower = mean - 2 * std
            upper = mean + 2 * std
            ax.errorbar(x+(i-offset)*width, mean, yerr=[mean - lower, upper - mean], color="gray", linestyle="none")

def plot_calibration_curve_relplot(ax, y_true, y_pred, color="tab:blue", labelfmt=".3f", labelsize=None):
    # reference: https://github.com/apple/ml-calibration/blob/18ff21a7e4e409fc4885690129f50211b32ea144/src/relplot/diagrams.py#L273
    def plot_rel_diagram(
        diagram,
        fig=None,
        ax=None,
        use_default_style=True,
        plot_main_line=True,
        simple_main_line=False,
        plot_density=False,
        plot_density_ticks=True,
        split_densities=False,
        color='red',
        density_color='gray',
        special_endpoints=False,
        endpoint_prob_thresh=0.01,
        plot_labels=True,
        plot_diagonal=True,
        **unused_kwargs,
    ):
        """
        Args:
            diagram (dict): Calibration data; result of relplot.prepare_rel_diagram
            fig (matplotlib.figure.Figure, optional): Figure to use, if provided. Defaults to None.
            ax (matplotlib.axes.Axes, optional): Axes to use, if provided. Defaults to None.
            use_default_style (bool, optional): Apply the recommended plot styling. Defaults to True.
            color (str, optional): Color to use for the main regression line. Defaults to 'red'.
            density_color (str, optional): Color to use for the density overlays. Defaults to 'gray'.
            plot_main_line (bool, optional): Plot the primary reliability curve, the regression function. Defaults to True.
            plot_confidence_band (bool, optional): Plot bootstrapped confidence bands around the main line. Defaults to True.
            plot_bag_lines (bool, optional): Plot the individual bootstrapped estimators. Defaults to False.
            num_bootstrap (int, optional): Number of bootstrap estimators. Defaults to 200.
            split_densities (bool, optional): Plot separate densities for f(x) on positive and negative labels, instead of a single density. Defaults to False.
            refined_kde (bool, optional): Use cross-validation to pick the kernel bandwidth for plotting the densities. Defaults to False.
            kde_bandwidth (float, optional): Override the default choice of bandwidth for plotting densities, if specified. Defaults to None.
            report_CE (bool, optional): Print the calibration error (smECE) on the plot. Defaults to True.
            report_CE_std (bool, optional): Compute and print a 95% confidence interval of the calibration error, estimated via bootstrapping. Defaults to True.
            custom_regressor (sklearn.base.BaseEstimator, optional): Use a custom sklearn estimator as the regressor, if specified. Defaults to None.

        Returns:
            matplotlib.figure.Figure: Reliability diagram figure.
        """
        import matplotlib as mpl # ADDED

        def get_ticks_from_density(dens, mesh, n_ticks=200):
            return np.random.default_rng(seed=0).choice(mesh, size=n_ticks, p=dens/np.sum(dens))

        def get_sizes_from_density(density):
            SIZE_SCALE = 200
            density /= 2
            filt = density < 1
            density_n = (density**2) * filt + np.sqrt(density) * (1-filt)
            return (density_n) * SIZE_SCALE


        # if use_default_style:
        #     set_default_style()

        color = np.array(mpl.colors.to_rgba(color))

        if ax is None or fig is None:
            fig, ax = plt.subplots(figsize=(6, 6))


        ## main diagram
        t = diagram["mesh"]
        if plot_diagonal:
            ax.plot(t, t, "k--", lw=1, alpha=0.2)

        density = diagram["density"]
        density_n = density / np.max(density)

        # ## bootstrapped lines (plot max 50 of them)
        # for mu in diagram["bags"]:
        #     alpha_scale = 0.20
        #     plot_line(
        #         t, mu, ax=ax, colors=[np.clip(color * di * alpha_scale, 0, 1) for di in density], lw=1
        #     )

        ## main red line (simple)
        if plot_main_line and simple_main_line:
            mu = diagram["mu"]
            # colors = [color * np.array([1, 1, 1, np.clip(di + 0.1, 0, 1)]) for di in density]
            # plot_line(t, mu, colors=colors, ax=ax, lw=2, ls="--")
            # ax.plot(t, mu, '--', color=color, clip_on=False)
            ax.plot(t, mu, '-', color=color, clip_on=False) # CHANGED

        ## main red line (density-dependent width)
        if plot_main_line and not simple_main_line:
            mesh_d = np.linspace(0, 1, 1000) # denser mesh for the line
            mu = diagram["mu"]
            mu_d = np.interp(mesh_d, diagram['mesh'], diagram['mu']) # denser mu
            density_d = np.interp(mesh_d, diagram['mesh'], diagram['density'])
            sizes = get_sizes_from_density(density_d)
            colors = [color * np.array([1, 1, 1, np.clip(di * 0.3, 0, 1)]) for di in density_d]
            ax.scatter(mesh_d, mu_d, marker='.', c=colors, s=sizes, clip_on=False, zorder=100)
            # linewidths = density_d * 50
            # plot_line(mesh_d, mu_d, ax=ax, colors=colors, linewidths=linewidths, clip_on=False, zorder=100)


        ## confidence bands
        if "upper" in diagram.keys():
            upper = diagram["upper"]
            lower = diagram["lower"]
            for i in range(len(t) - 1):
                ax.fill_between(
                    [t[i], t[i + 1]],
                    [lower[i], lower[i + 1]],
                    [upper[i], upper[i + 1]],
                    lw=0,
                    edgecolor=None,
                    # alpha = np.clip(density[i] * 0.5, 0, 1),
                    alpha = density_n[i] * 0.6,
                    color='gray',
                    clip_on=True
                )


        ## endpoint densities
        if special_endpoints:
            for x0 in [0, 1]:
                mu = diagram[f'{x0}pt_mu']
                pt_dens = diagram[f'{x0}pt_density']
                if pt_dens < endpoint_prob_thresh:
                    continue # Don't plot small probabilities

                msize = 100 * (pt_dens)
                lw = msize / 3
                ax.plot([x0], [mu], color=color, clip_on=False, zorder=100, marker='o', markersize=msize,markerfacecolor=color, markeredgecolor='none', alpha=1)
                if f'{x0}pt_upper' in diagram.keys():
                    ub = diagram[f'{x0}pt_upper']
                    lb = diagram[f'{x0}pt_lower']
                    _, _, lines = ax.errorbar(x0, mu, yerr=[[mu-lb], [ub-mu]], clip_on=False, zorder=100, capsize=0, marker='none', lw=lw, color=color) # capsize=3
                    lines[0].set_capstyle('round')

                ha = 'left' if x0 == 0 else 'right'
                xform = ax.transData.inverted()
                width_pt = (xform.transform((msize, 0)) - xform.transform((0, 0)))[0] / 2.0
                text_shift = (0.01 + width_pt ) * (1 if x0 == 0 else -1)
                ax.text(x=x0 + text_shift, y=mu, s=f'({pt_dens:.2f})', fontsize='x-small', va='center', ha=ha, clip_on=False)

        # if plot_labels:
        #     if config.use_tex_fonts:
        #         ax.set_xlabel("$f$")
        #         ax.set_ylabel(r"$\mathbb{E}[ y \mid f ]$")
        #     else:
        #         ax.set_xlabel("f")
        #         ax.set_ylabel(r"E[ y | f ]")


        ax.set_xlim((0, 1))
        ax.set_ylim((0, 1))
        ax.set_aspect('equal')

        ## density subplot
        if plot_density:
            densities = diagram["densities"] if split_densities else [diagram["density"]]
            max_dens = np.max(densities) # CHANGED
            alpha = 0.2
            for i, dens in enumerate(densities):
                if not split_densities:
                    hatch = None
                else:
                    hatch = "//" if i == 0 else "\\\\"
                max_density_height = 0.3
                height = dens / max_dens * max_density_height # CHANGED
                ax.fill_between(
                    t,
                    np.zeros_like(t),
                    height,
                    alpha=alpha,
                    # color=density_color,
                    color=density_color[i] if split_densities else density_color, # CHANGED
                    # hatch=hatch, # CHANGED
                    label=f"y={i}",
                )
        if plot_density_ticks:
            def _plot_ticks(fi, height=0.01, shift=0.0):
                # fi = get_ticks_from_density(dens, t, n_ticks = n_ticks)
                ax.vlines(fi, -height + shift, +height + shift, color='black', clip_on=False, zorder = 101)

            n_ticks = 100
            tickH = 0.01
            fs, ys = diagram['f_samp'][:n_ticks], diagram['y_samp'][:n_ticks]
            _plot_ticks(fs[ys == 0], tickH, -tickH)
            _plot_ticks(fs[ys == 1], tickH, +tickH)

        if "ce" in diagram.keys():
            ice = diagram["ce"]
            if "ce_ci_width" in diagram.keys():
                wid = diagram["ce_ci_width"]
                ax.text(0.05, 0.9, f"$\\mathrm{{smECE}}: {ice:.3f}\\pm {wid:.3f}$", zorder=1000)
            else:
                ax.text(0.05, 0.9, f"smECE: {ice:.3f}", zorder=1000)
        return fig, ax

    diagram = rp.prepare_rel_diagram(y_pred, y_true)
    ice = diagram.pop("ce")
    plot_rel_diagram(
        diagram,
        fig=ax.get_figure(),
        ax=ax,
        use_default_style=False,
        plot_labels=False,
        simple_main_line=True,
        color=color,
        plot_density_ticks=False,
        plot_density=True,
        split_densities=True,
        density_color=["tab:red", "tab:green"],
    )
    plot_annotation(ax, f"$\\mathrm{{smECE}}: {ice:{labelfmt}}$", fontsize=labelsize)

def annotate_agg_over(ax, agg_over):
    ax.text(1.0, 1.02, f"agg. over {agg_over}", ha="right", va="bottom", fontsize=FONTSIZE_SMALL, transform=ax.transAxes)

# plotting functions

def make_plots_datasets(y_true_all, y_pred_all, agg_over, dataset_names=None, model_names=None, method_names=None, sort_by=None, label_rotation=45, n_bins=20):
    num_seeds = len(y_true_all)
    dataset_names, model_names, method_names = detect_names_from_dict(y_true_all[0], dataset_names=dataset_names, model_names=model_names, method_names=method_names)

    # compute
    scores = defaultdict(lambda: np.zeros((len(dataset_names), num_seeds)))
    for i, dataset_name in enumerate(dataset_names):
        for j, seed in enumerate(y_true_all):
            y_true = np.array(aggregate_responses(y_true_all[seed], dataset_name, model_names, method_names))
            y_pred = np.array(aggregate_responses(y_pred_all[seed], dataset_name, model_names, method_names))

            ece, _ = calibration_error(y_true, y_pred, n_bins=n_bins)
            scores["ece"][i, j] = ece
            scores["smECE"][i, j] = rp.smECE(y_pred, y_true)
            scores["accuracy"][i, j] = np.mean(y_true)
            scores["confidence"][i, j] = np.mean(y_pred)
            scores["confidence_n_distinct"][i, j] = len(np.unique(y_pred))
            scores["confidence_variance"][i, j] = np.std(y_pred)
            scores["kl_div_over_dataset"][i, j] = np.mean([
                kl_div(
                    empirical_distr(aggregate_responses(y_pred_all[seed], dataset_name, model_name, method_name), n_bins),
                    empirical_distr(aggregate_responses(y_pred_all[seed], dataset_names, model_name, method_name), n_bins),
                )
                for model_name in model_names
                for method_name in method_names
            ])

    # sort
    if sort_by is not None:
        scores_zipped = zip(scores[sort_by], dataset_names, scores["accuracy"], scores["confidence"], scores["ece"], scores["smECE"], scores["confidence_n_distinct"], scores["confidence_variance"], scores["kl_div_over_dataset"])
        scores_zipped = sorted(scores_zipped, key=lambda x: x.mean(), reverse=True)
        _, dataset_names, scores["accuracy"], scores["confidence"], scores["ece"], scores["smECE"], scores["confidence_n_distinct"], scores["confidence_variance"], scores["kl_div_over_dataset"] = zip(*scores_zipped)

    # plot
    x = np.arange(len(dataset_names))

    fig1, ax = Plotter.create()
    Plotter.set(
        ax,
        ylim=YLIM_CALIBRATION,
        xticks=dict(ticks=x, labels=translate_dataset(dataset_names), rotation=label_rotation, ha="right"),
    )
    plot_grouped_bar([
        (ax, scores["accuracy"], dict(label="accuracy", color=COLOR_ACCURACY)),
        (ax, scores["confidence"], dict(label="confidence", color=COLOR_CONFIDENCE)),
        (ax, scores["ece"], dict(label="ECE", color=COLOR_ECE)),
        (ax, scores["smECE"], dict(label="smECE", color=COLOR_SMECE)),
    ], with_lines=True, with_errorbars=True)
    annotate_agg_over(ax, agg_over)

    fig2, ax1 = Plotter.create()
    ax2 = ax1.twinx()
    Plotter.set(
        ax1,
        ylim=YLIM_OTHERS_1,
        xticks=dict(ticks=x, labels=translate_dataset(dataset_names), rotation=label_rotation, ha="right"),
    )
    Plotter.set(
        ax2,
        ylim=YLIM_OTHERS_2,
    )
    plot_grouped_bar([
        (ax1, scores["confidence_n_distinct"], dict(label="# distinct", color=COLOR_CONF_N_DISTINCT)),
        (ax2, scores["confidence_variance"], dict(label="variance", color=COLOR_CONF_VARIANCE)),
        (ax2, scores["kl_div_over_dataset"], dict(label="kl_div", color=COLOR_KL_DIV)),
    ], with_lines=True, with_errorbars=True)
    annotate_agg_over(ax1, agg_over)

    return fig1, fig2

def make_plots_models(y_true_all, y_pred_all, agg_over, model_names=None, method_names=None, sort_by=None, label_rotation=45, n_bins=20):
    if model_names is not None:
        model_names, none_indices = extract_none_indices(model_names)
    else:
        none_indices = []

    num_seeds = len(y_true_all)
    dataset_names, model_names, method_names = detect_names_from_dict(y_true_all[0], model_names=model_names, method_names=method_names)

    # compute
    scores = defaultdict(lambda: np.zeros((len(model_names), num_seeds)))
    for i, model_name in enumerate(model_names):
        for j, seed in enumerate(y_true_all):
            y_true = np.array(aggregate_responses(y_true_all[seed], dataset_names, model_name, method_names))
            y_pred = np.array(aggregate_responses(y_pred_all[seed], dataset_names, model_name, method_names))

            ece, _ = calibration_error(y_true, y_pred, n_bins=n_bins)
            scores["ece"][i, j] = ece
            scores["smECE"][i, j] = rp.smECE(y_pred, y_true)
            scores["accuracy"][i, j] = np.mean(y_true)
            scores["confidence"][i, j] = np.mean(y_pred)
            scores["confidence_n_distinct"][i, j] = len(np.unique(y_pred))
            scores["confidence_variance"][i, j] = np.std(y_pred)
            scores["kl_div_over_dataset"][i, j] = np.mean([
                kl_div(
                    empirical_distr(aggregate_responses(y_pred_all[seed], dataset_name, model_name, method_name), n_bins),
                    empirical_distr(aggregate_responses(y_pred_all[seed], dataset_names, model_name, method_name), n_bins),
                )
                for dataset_name in dataset_names
                for method_name in method_names
            ])

    # sort
    if sort_by is not None:
        scores_zipped = zip(scores[sort_by], model_names, scores["accuracy"], scores["confidence"], scores["ece"], scores["smECE"], scores["confidence_n_distinct"], scores["confidence_variance"], scores["kl_div_over_dataset"])
        scores_zipped = sorted(scores_zipped, key=lambda x: x.mean(), reverse=True)
        _, model_names, scores["accuracy"], scores["confidence"], scores["ece"], scores["smECE"], scores["confidence_n_distinct"], scores["confidence_variance"], scores["kl_div_over_dataset"] = zip(*scores_zipped)

    # plot
    x = np.arange(len(model_names), dtype=float)

    fig1, ax = Plotter.create()
    Plotter.set(
        ax,
        ylim=YLIM_CALIBRATION,
        xticks=dict(ticks=x, labels=translate_model(model_names), rotation=label_rotation, ha="right"),
    )
    plot_grouped_bar([
        (ax, scores["accuracy"], dict(label="accuracy", color=COLOR_ACCURACY, fmt="{:.2f}")),
        (ax, scores["confidence"], dict(label="confidence", color=COLOR_CONFIDENCE, fmt="{:.2f}")),
        (ax, scores["ece"], dict(label="ECE", color=COLOR_ECE, fmt="{:.2f}")),
        (ax, scores["smECE"], dict(label="smECE", color=COLOR_SMECE, fmt="{:.2f}")),
    ], with_lines=True, with_errorbars=True, none_indices=none_indices)
    annotate_agg_over(ax, agg_over)

    fig2, ax1 = Plotter.create()
    ax2 = ax1.twinx()
    Plotter.set(
        ax1,
        ylim=YLIM_OTHERS_1,
        xticks=dict(ticks=x, labels=translate_model(model_names), rotation=label_rotation, ha="right"),
    )
    Plotter.set(
        ax2,
        ylim=YLIM_OTHERS_2,
    )
    plot_grouped_bar([
        (ax1, scores["confidence_n_distinct"], dict(label="# distinct", color=COLOR_CONF_N_DISTINCT)),
        (ax2, scores["confidence_variance"], dict(label="variance", color=COLOR_CONF_VARIANCE)),
        (ax2, scores["kl_div_over_dataset"], dict(label="kl_div", color=COLOR_KL_DIV)),
    ], with_lines=True, with_errorbars=True, none_indices=none_indices)
    annotate_agg_over(ax1, agg_over)

    return fig1, fig2

def make_plots_methods(y_true_all, y_pred_all, agg_over, model_names=None, method_names=None, label_rotation=45, single_plot=False, n_bins=20):
    if method_names is not None:
        method_names, none_indices = extract_none_indices(method_names)
    else:
        none_indices = []

    num_seeds = len(y_true_all)
    dataset_names, model_names, method_names = detect_names_from_dict(
        y_true_all[0],
        model_names=model_names,
        method_names=method_names,
    )

    # compute
    scores = defaultdict(lambda: np.zeros((len(method_names), num_seeds)))
    for i, method_name in enumerate(method_names):
        for j, seed in enumerate(y_true_all):
            y_true = np.array(aggregate_responses(y_true_all[seed], dataset_names, model_names, method_name))
            y_pred = np.array(aggregate_responses(y_pred_all[seed], dataset_names, model_names, method_name))

            ece, _ = calibration_error(y_true, y_pred, n_bins=n_bins)
            scores["ece"][i, j] = ece
            scores["smECE"][i, j] = rp.smECE(y_pred, y_true)
            scores["accuracy"][i, j] = np.mean(y_true)
            scores["confidence"][i, j] = np.mean(y_pred)
            scores["confidence_n_distinct"][i, j] = len(np.unique(y_pred))
            scores["confidence_variance"][i, j] = np.std(y_pred)
            scores["kl_div_over_dataset"][i, j] = np.mean([
                kl_div(
                    empirical_distr(aggregate_responses(y_pred_all[seed], dataset_name, model_name, method_name), n_bins),
                    empirical_distr(aggregate_responses(y_pred_all[seed], dataset_names, model_name, method_name), n_bins),
                )
                for dataset_name in dataset_names
                for model_name in model_names
            ])

    # plot
    x = np.arange(len(method_names), dtype=float)

    fig1, ax = Plotter.create()
    fig2, ax1 = Plotter.create()
    ax2 = ax1.twinx()

    if single_plot:
        Plotter.set(ax, xticks=[])
    else:
        Plotter.set(ax, xticks=dict(ticks=x, labels=translate_method(method_names), rotation=label_rotation, ha="right"))
    Plotter.set(
        ax,
        ylim=YLIM_CALIBRATION,
    )
    plot_grouped_bar([
        (ax, scores["accuracy"], dict(label="accuracy", color=COLOR_ACCURACY, fmt="{:.2f}")),
        (ax, scores["confidence"], dict(label="confidence", color=COLOR_CONFIDENCE, fmt="{:.2f}")),
        (ax, scores["ece"], dict(label="ECE", color=COLOR_ECE, fmt="{:.2f}")),
        (ax, scores["smECE"], dict(label="smECE", color=COLOR_SMECE, fmt="{:.2f}")),
    ], with_labels=True, with_lines=True, with_errorbars=True, none_indices=none_indices)
    annotate_agg_over(ax, agg_over)

    Plotter.set(
        ax1,
        ylim=YLIM_OTHERS_1,
        xticks=dict(ticks=x, labels=translate_method(method_names), rotation=label_rotation, ha="right"),
    )
    Plotter.set(
        ax2,
        ylim=YLIM_OTHERS_2,
    )
    plot_grouped_bar([
        (ax1, scores["confidence_n_distinct"], dict(label="n_distinct", color=COLOR_CONF_N_DISTINCT, fmt="{:.0f}")),
        (ax2, scores["confidence_variance"], dict(label="variance", color=COLOR_CONF_VARIANCE, fmt="{:.2f}")),
        (ax2, scores["kl_div_over_dataset"], dict(label="kl_div", color=COLOR_KL_DIV, fmt="{:.2f}")),
    ], with_labels=True, with_lines=True, with_errorbars=True, none_indices=none_indices)
    if not single_plot:
        annotate_agg_over(ax1, agg_over)

    return fig1, fig2

In [ ]:
Plotter.configure(save_always=True)

### Main plots

In [ ]:
def plot(y_true_all, y_pred_all):
    dataset_names_all = [
        "sciq",
        "arc-e",
        "arc-c",
        "commonsense_qa",
        "social_i_qa",
        "mmlu",
        "trivia_qa",
        "truthful_qa-mc1",
        "truthful_qa-mc2",
        "logi_qa",
    ]
    model_names_all = [
        "gemma1.1-2b-it",
        "gemma1.1-7b-it",
        None,
        "llama3-8b-instruct",
        "llama3-70b-instruct",
        None,
        "qwen1.5-7b-chat",
        "qwen1.5-32b-chat",
        "qwen1.5-72b-chat",
        "qwen1.5-110b-chat",
        None,
        "gpt3.5-turbo",
        "gpt4o-mini",
        "gpt4o",
    ]

    plots = []

    fig1, fig2 = make_plots_datasets(y_true_all, y_pred_all, "models[all], methods[all]", dataset_names=dataset_names_all, label_rotation=35)
    fig1.axes[0].legend(loc="center left")
    plots.append((fig1, "datasets-calibration"))
    # plots.append((fig2, "datasets-others"))

    fig1, fig2 = make_plots_models(y_true_all, y_pred_all, "datasets[all], methods[all]", model_names=model_names_all)
    plots.append((fig1, "models-calibration"))
    # plots.append((fig2, "models-others"))

    Plotter.finish(
        plots, figwidth=0.5,
        grid_ncols=2, consistent_size=True,
        # save=True,
    )

plot(y_true_all, y_pred_all)

In [ ]:
def plot(y_true_all, y_pred_all):
    plots = []

    fig1, fig2 = make_plots_methods(y_true_all, y_pred_all, "datasets[all], models[tiny]", model_names=MODELS_TINY, method_names=METHODS_COMBO, label_rotation=25, single_plot=True)
    h1, l1 = fig2.axes[0].get_legend_handles_labels()
    h2, l2 = fig2.axes[1].get_legend_handles_labels()
    fig2.axes[0].legend(h1+h2[:1], l1+l2[:1], loc="upper left")
    fig2.axes[1].legend(h2[1:], l2[1:], loc="upper right")
    plots.append((fig1, "methods-calibration-combo-tinymodels"))
    plots.append((fig2, "methods-others-combo-tinymodels"))

    fig1, fig2 = make_plots_methods(y_true_all, y_pred_all, "datasets[all], models[large]", model_names=MODELS_LARGE, method_names=METHODS_COMBO, label_rotation=25, single_plot=True)
    fig1.axes[0].legend(loc="center right")
    plots.append((fig1, "methods-calibration-combo-largemodels"))
    plots.append((fig2, "methods-others-combo-largemodels"))
    plots[::2], plots[1::2] = plots[:2], plots[2:]

    Plotter.finish(
        plots, figwidth=0.5,
        grid_ncols=2, consistent_size=True,
        # save=True,
    )

plot(y_true_all, y_pred_all)

In [ ]:
def plot(y_true_all, y_pred_all):
    def make_plot(y_true_all, y_pred_all, agg_over, model_name, method_names):
        dataset_names, _, _ = detect_names_from_dict(y_true_all)

        fig, axes = Plotter.create(ncols=len(method_names), sharey=True)
        fig.supxlabel("confidence")
        axes[0].set_ylabel("accuracy")

        for ax, method_name in zip(axes, method_names):
            y_true = aggregate_responses(y_true_all, dataset_names, model_name, method_name)
            y_pred = aggregate_responses(y_pred_all, dataset_names, model_name, method_name)
            plot_calibration_curve(ax, y_true, y_pred, n_bins=20, labelfmt="{:.2f}", labelsize=FONTSIZE_DEFAULT)

        annotate_agg_over(axes[-1], agg_over)

        return fig

    plots = []

    fig = make_plot(y_true_all, y_pred_all, "datasets[all]", "llama3-8b-instruct", ["basic_1s", "combo_1s_v2"])
    plots.append((fig, "methods-calibration_curve-basic_vs_combo-llama3_8b"))
    fig = make_plot(y_true_all, y_pred_all, "datasets[all]", "gpt4o", ["basic_1s", "combo_1s_v2"])
    plots.append((fig, "methods-calibration_curve-basic_vs_combo-gpt4o"))

    Plotter.finish(
        plots, figwidth=0.5, axratio=1,
        consistent_size=True,
        # save=True,
    )

plot(y_true_all[0], y_pred_all[0])

### Appendix plots

In [ ]:
def plot(responses_all):
    dataset_names, model_names, method_names = detect_names_from_dict(responses_all)

    # compute
    answer_stats = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: (0, 0, 0))))
    answer_stats_over_dataset = defaultdict(lambda: defaultdict(lambda: (0, 0, 0)))
    for model_name in model_names:
        for method_name in method_names:
            for dataset_name in dataset_names:
                responses = responses_all[dataset_name][model_name][method_name]
                if responses is None:
                    continue
                answer_stats[model_name][method_name][dataset_name] = (
                    len(responses[VALID_ANSWER]),
                    len(responses[NO_ANSWER]),
                    len(responses[INVALID_ANSWER]),
                )
            answer_stats_over_dataset[model_name][method_name] = (
                sum(answer_stats[model_name][method_name][dataset_name][0] for dataset_name in dataset_names),
                sum(answer_stats[model_name][method_name][dataset_name][1] for dataset_name in dataset_names),
                sum(answer_stats[model_name][method_name][dataset_name][2] for dataset_name in dataset_names),
            )

    percentage_valid_answer = np.zeros((len(method_names), len(model_names)))
    percentage_no_answer = np.zeros((len(method_names), len(model_names)))
    for i, method_name in enumerate(method_names):
        for j, model_name in enumerate(model_names):
            n_valid_answer, n_no_answer, n_invalid_answer = answer_stats_over_dataset[model_name][method_name]
            n_total = n_valid_answer + n_no_answer + n_invalid_answer
            percentage_valid_answer[i, j] = n_valid_answer / n_total if n_total > 0 else None
            percentage_no_answer[i, j] = n_no_answer / n_total if n_total > 0 else None

    plots = []

    fig, ax = Plotter.create()
    plot_heatmap(ax, percentage_valid_answer, translate_method(method_names), translate_model(model_names), plot_mean=False, format="{:.2f}", vmin=0, vmax=1.3, cmap="Greens")
    plots.append((fig, "answer_statistics"))

    # fig, ax = Plotter.create()
    # plot_heatmap(ax, percentage_no_answer, method_names, model_names, plot_mean=False, format="{:.2f}", vmin=0, vmax=0.3, cmap="Reds")
    # plots.append((fig, "answer_statistics-no_answers"))

    Plotter.finish(
        plots, figwidth=0.59, axratio=len(method_names) / len(model_names),
        # save=True,
    )

plot(responses_all)

In [ ]:
def plot(y_true_all, y_pred_all):
    dataset_names_all = [
        "sciq",
        "arc-e",
        "arc-c",
        "commonsense_qa",
        "social_i_qa",
        "mmlu",
        "trivia_qa",
        "truthful_qa-mc1",
        "truthful_qa-mc2",
        "logi_qa",
    ]

    plots = []

    fig1, fig2 = make_plots_datasets(y_true_all, y_pred_all, "models[tiny], methods[all]", dataset_names=dataset_names_all, model_names=MODELS_TINY, label_rotation=35)
    fig1.axes[0].legend(loc="center left")
    plots.append((fig1, "datasets-calibration-tinymodels"))
    # plots.append((fig2, "datasets-others-tinymodels"))

    fig1, fig2 = make_plots_datasets(y_true_all, y_pred_all, "models[large], methods[all]", dataset_names=dataset_names_all, model_names=MODELS_LARGE, label_rotation=35)
    plots.append((fig1, "datasets-calibration-largemodels"))
    # plots.append((fig2, "datasets-others-largemodels"))

    Plotter.finish(
        plots, figwidth=0.5,
        grid_ncols=2, consistent_size=True,
        # save=True,
    )

plot(y_true_all, y_pred_all)

In [ ]:
def plot(y_true_all, y_pred_all):
    dataset_names, _, method_names = detect_names_from_dict(y_true_all)

    fig, ax = Plotter.create()
    ax.set_xlabel("confidence")
    ax.set_ylabel("accuracy")

    y_true = aggregate_responses(y_true_all, dataset_names, "gemma1.1-2b-it", method_names)
    y_pred = aggregate_responses(y_pred_all, dataset_names, "gemma1.1-2b-it", method_names)
    plot_calibration_curve(ax, y_true, y_pred, n_bins=20, labelfmt="{:.2f}", labelsize=FONTSIZE_DEFAULT)
    annotate_agg_over(ax, "datasets[all], methods[all]")

    Plotter.finish(
        (fig, "models-calibration_curve-gemma1_1_2b"), figwidth=0.4, axratio=1,
        # save=True,
    )

plot(y_true_all[0], y_pred_all[0])

In [ ]:
def plot(y_true_all, y_pred_all):
    def make_plots_all(model_names, agg_over, suffix=""):
        plots = []

        fig1, fig2 = make_plots_methods(y_true_all, y_pred_all, agg_over, model_names=model_names, method_names=METHODS_SCORE_RANGE, label_rotation=30, single_plot=True)
        plots.append((fig1, f"methods-calibration-score_range{suffix}"))
        plots.append((fig2, f"methods-others-score_range{suffix}"))
        fig1, fig2 = make_plots_methods(y_true_all, y_pred_all, agg_over, model_names=model_names, method_names=METHODS_SCORE_FORMULATION, label_rotation=30, single_plot=True)
        plots.append((fig1, f"methods-calibration-score_formulation{suffix}"))
        plots.append((fig2, f"methods-others-score_formulation{suffix}"))
        fig1, fig2 = make_plots_methods(y_true_all, y_pred_all, agg_over, model_names=model_names, method_names=METHODS_ADVANCED_FORMULATION, label_rotation=30, single_plot=True)
        plots.append((fig1, f"methods-calibration-advanced_formulation{suffix}"))
        plots.append((fig2, f"methods-others-advanced_formulation{suffix}"))
        fig1, fig2 = make_plots_methods(y_true_all, y_pred_all, agg_over, model_names=model_names, method_names=METHODS_FEW_SHOT, label_rotation=30, single_plot=True)
        plots.append((fig1, f"methods-calibration-few_shot{suffix}"))
        plots.append((fig2, f"methods-others-few_shot{suffix}"))
        fig1, fig2 = make_plots_methods(y_true_all, y_pred_all, agg_over, model_names=model_names, method_names=METHODS_OTHERS, label_rotation=30, single_plot=True)
        plots.append((fig1, f"methods-calibration-others{suffix}"))
        plots.append((fig2, f"methods-others-others{suffix}"))

        return plots

    plots_tiny = make_plots_all(MODELS_TINY, "datasets[all], models[tiny]", suffix="-tinymodels")
    for (fig1, _), (fig2, _) in zip(plots_tiny[::2], plots_tiny[1::2]):
        h1, l1 = fig2.axes[0].get_legend_handles_labels()
        h2, l2 = fig2.axes[1].get_legend_handles_labels()
        fig2.axes[0].legend(h1+h2[:1], l1+l2[:1], loc="upper left")
        fig2.axes[1].legend(h2[1:], l2[1:], loc="upper right")

    plots_large = make_plots_all(MODELS_LARGE, "datasets[all], models[large]", suffix="-largemodels")
    for (fig1, _), (fig2, _) in zip(plots_large[::2], plots_large[1::2]):
        fig1.axes[0].legend(loc="center right")

    plots = plots_tiny + plots_large
    plots[::2], plots[1::2] = plots[:10], plots[10:]

    Plotter.finish(
        plots, figwidth=0.5,
        grid_ncols=2, consistent_size=True,
        # save=True,
    )

plot(y_true_all, y_pred_all)

In [ ]:
def plot(y_true_all, y_pred_all):
    def make_plot(y_true_all, y_pred_all, agg_over, model_name, method_names):
        dataset_names, _, _ = detect_names_from_dict(y_true_all)

        fig, axes = Plotter.create(ncols=len(method_names), sharey=True)
        fig.supxlabel("confidence")
        axes[0].set_ylabel("accuracy")

        for ax, method_name in zip(axes, method_names):
            y_true = aggregate_responses(y_true_all, dataset_names, model_name, method_name)
            y_pred = aggregate_responses(y_pred_all, dataset_names, model_name, method_name)
            plot_calibration_curve_relplot(ax, y_true, y_pred, labelfmt=".2f", labelsize=FONTSIZE_DEFAULT)

        annotate_agg_over(axes[-1], agg_over)

        return fig

    plots = []

    fig = make_plot(y_true_all, y_pred_all, "datasets[all]", "llama3-8b-instruct", ["basic_1s", "combo_1s_v2"])
    plots.append((fig, "methods-calibration_curve_relplot-basic_vs_combo-llama3_8b"))
    fig = make_plot(y_true_all, y_pred_all, "datasets[all]", "gpt4o", ["basic_1s", "combo_1s_v2"])
    plots.append((fig, "methods-calibration_curve_relplot-basic_vs_combo-gpt4o"))

    Plotter.finish(
        plots, figwidth=0.5, axratio=1,
        consistent_size=True,
        # save=True,
    )

plot(y_true_all[0], y_pred_all[0])

In [ ]:
def plot_grouped_bar2(i, n_bars, axes_scores_args, width=None, alpha=None, with_labels=False, with_lines=False, with_errorbars=False, none_indices=[]):
    # def filter_singles(l):
    #     l_new = []
    #     for i in range(len(l)):
    #         if not np.isnan(l[i]):
    #             left_is_val = i-1 >= 0 and not np.isnan(l[i-1])
    #             right_is_val = i+1 < len(l) and not np.isnan(l[i+1])
    #             if left_is_val:
    #                 l_new.append(l[i])
    #             elif right_is_val:
    #                 if len(l_new) > 0:
    #                     l_new.append(np.nan)
    #                 l_new.append(l[i])
    #     return np.asarray(l_new)

    n_scores = len(axes_scores_args[0][1])
    if width is None:
        width = 1 / (n_bars + 1.5)
    if alpha is None:
        alpha = 0.5 if with_lines else 1

    x = np.arange(n_scores, dtype=float)
    offset = (n_bars - 1) / 2
    # plot bars
    mean_prev = 0
    for ax, scores, args in axes_scores_args:
        mean = np.mean(scores, axis=1)
        if with_errorbars:
            std = np.std(scores, axis=1)
            lower = mean - 2 * std
            upper = mean + 2 * std
            yerr = [mean - lower, upper - mean]
        else:
            yerr = None
        fmt = args.pop("fmt", "{:.2f}")
        bar_container = ax.bar(x+(i-offset)*width, mean, width, bottom=mean_prev, yerr=yerr, alpha=alpha, **args)
        mean_prev += mean
        if with_labels:
            ax: plt.Axes = ax
            ax.bar_label(bar_container, fmt=fmt, fontsize=FONTSIZE_SMALL, label_type="center")
    # # plot lines
    # if with_lines:
    #     x_ = filter_singles(np.insert(x, none_indices, np.nan))
    #     for i, (ax, scores, args) in enumerate(axes_scores_args):
    #         mean = np.mean(scores, axis=1)
    #         ax.plot(x_+(i-offset)*width, filter_singles(np.insert(mean, none_indices, np.nan)), color=args["color"], marker="o", markersize=2)
    # # plot errorbars (again to appear on top of lines)
    # if with_errorbars:
    #     for i, (ax, scores, args) in enumerate(axes_scores_args):
    #         mean = np.mean(scores, axis=1)
    #         std = np.std(scores, axis=1)
    #         lower = mean - 2 * std
    #         upper = mean + 2 * std
    #         ax.errorbar(x+(i-offset)*width, mean, yerr=[mean - lower, upper - mean], color="gray", linestyle="none")

def make_plots_datasets2(y_true_all, y_pred_all, agg_over, dataset_names=None, model_names=None, method_names=None, sort_by=None, label_rotation=45, n_bins=20):
    num_seeds = len(y_true_all)
    dataset_names, model_names, method_names = detect_names_from_dict(y_true_all[0], dataset_names=dataset_names, model_names=model_names, method_names=method_names)

    # compute
    scores = defaultdict(lambda: np.zeros((len(dataset_names), num_seeds)))
    for i, dataset_name in enumerate(dataset_names):
        for j, seed in enumerate(y_true_all):
            y_true = np.array(aggregate_responses(y_true_all[seed], dataset_name, model_names, method_names))
            y_pred = np.array(aggregate_responses(y_pred_all[seed], dataset_name, model_names, method_names))

            ece, _ = calibration_error(y_true, y_pred, n_bins=n_bins)
            cal, res, unc = brier_score_decomposition(y_true, y_pred, n_bins=n_bins)
            scores["ece"][i, j] = ece
            # scores["smECE"][i, j] = rp.smECE(np.array(y_pred), np.array(y_true))
            scores["brier_score"][i, j] = np.mean((y_true - y_pred) ** 2)
            scores["brier_score_cal"][i, j] = cal
            scores["brier_score_res"][i, j] = res
            scores["brier_score_unc"][i, j] = unc
            # scores["log_loss"][i, j] = -np.mean(y_true * np.log(y_pred + 1e-15) + (1 - y_true) * np.log(1 - y_pred + 1e-15))

    # plot
    x = np.arange(len(dataset_names))

    fig1, ax = Plotter.create()
    Plotter.set(
        ax,
        ylim=YLIM_CALIBRATION,
        xticks=dict(ticks=x, labels=translate_dataset(dataset_names), rotation=label_rotation, ha="right"),
    )
    plot_grouped_bar([
        (ax, scores["ece"], dict(label="ECE", color=COLOR_ECE, fmt="{:.2f}")),
        (ax, scores["brier_score"], dict(color=COLOR_BRIER, fmt="{:.2f}", alpha=0)),
    ], with_lines=True, with_errorbars=True)
    plot_grouped_bar2(1, 2, [
        (ax, scores["brier_score_cal"], dict(label="Brier score (calibration)", color=COLOR_SMECE, fmt="{:.2f}")),
        (ax, -scores["brier_score_res"], dict(label="Brier score (resolution)", color=COLOR_CONF_VARIANCE, fmt="{:.2f}")),
        (ax, scores["brier_score_unc"], dict(label="Brier score (uncertainty)", color="tab:gray", fmt="{:.2f}")),
    ], with_lines=True, with_errorbars=False)
    annotate_agg_over(ax, agg_over)

    return fig1

def make_plots_models2(y_true_all, y_pred_all, agg_over, model_names=None, method_names=None, sort_by=None, label_rotation=45, n_bins=20):
    if model_names is not None:
        model_names, none_indices = extract_none_indices(model_names)
    else:
        none_indices = []

    num_seeds = len(y_true_all)
    dataset_names, model_names, method_names = detect_names_from_dict(y_true_all[0], model_names=model_names, method_names=method_names)

    # compute
    scores = defaultdict(lambda: np.zeros((len(model_names), num_seeds)))
    for i, model_name in enumerate(model_names):
        for j, seed in enumerate(y_true_all):
            y_true = np.array(aggregate_responses(y_true_all[seed], dataset_names, model_name, method_names))
            y_pred = np.array(aggregate_responses(y_pred_all[seed], dataset_names, model_name, method_names))

            ece, _ = calibration_error(y_true, y_pred, n_bins=n_bins)
            cal, res, unc = brier_score_decomposition(y_true, y_pred, n_bins=n_bins)
            scores["ece"][i, j] = ece
            # scores["smECE"][i, j] = rp.smECE(np.array(y_pred), np.array(y_true))
            scores["brier_score"][i, j] = np.mean((y_true - y_pred) ** 2)
            scores["brier_score_cal"][i, j] = cal
            scores["brier_score_res"][i, j] = res
            scores["brier_score_unc"][i, j] = unc
            # scores["log_loss"][i, j] = -np.mean(y_true * np.log(y_pred + 1e-15) + (1 - y_true) * np.log(1 - y_pred + 1e-15))

    # plot
    x = np.arange(len(model_names), dtype=float)

    fig1, ax = Plotter.create()
    Plotter.set(
        ax,
        ylim=YLIM_CALIBRATION,
        xticks=dict(ticks=x, labels=translate_model(model_names), rotation=label_rotation, ha="right"),
    )
    plot_grouped_bar([
        (ax, scores["ece"], dict(label="ECE", color=COLOR_ECE, fmt="{:.2f}")),
        (ax, scores["brier_score"], dict(color=COLOR_BRIER, fmt="{:.2f}", alpha=0)),
    ], with_lines=True, with_errorbars=True, none_indices=none_indices)
    plot_grouped_bar2(1, 2, [
        (ax, scores["brier_score_cal"], dict(label="Brier score (calibration)", color=COLOR_SMECE, fmt="{:.2f}")),
        (ax, -scores["brier_score_res"], dict(label="Brier score (resolution)", color=COLOR_CONF_VARIANCE, fmt="{:.2f}")),
        (ax, scores["brier_score_unc"], dict(label="Brier score (uncertainty)", color="tab:gray", fmt="{:.2f}")),
    ], with_lines=True, with_errorbars=False, none_indices=none_indices)
    annotate_agg_over(ax, agg_over)

    return fig1

def make_plots_methods2(y_true_all, y_pred_all, agg_over, model_names=None, method_names=None, label_rotation=45, single_plot=False, n_bins=20, hack=False):
    if method_names is not None:
        method_names, none_indices = extract_none_indices(method_names)
    else:
        none_indices = []

    num_seeds = len(y_true_all)
    dataset_names, model_names, method_names = detect_names_from_dict(
        y_true_all[0],
        model_names=model_names,
        method_names=method_names,
    )

    # compute
    scores = defaultdict(lambda: np.zeros((len(method_names), num_seeds)))
    for i, method_name in enumerate(method_names):
        for j, seed in enumerate(y_true_all):
            y_true = np.array(aggregate_responses(y_true_all[seed], dataset_names, model_names, method_name))
            y_pred = np.array(aggregate_responses(y_pred_all[seed], dataset_names, model_names, method_name))

            ece, _ = calibration_error(y_true, y_pred, n_bins=n_bins)
            cal, res, unc = brier_score_decomposition(y_true, y_pred, n_bins=n_bins)
            scores["ece"][i, j] = ece
            # scores["smECE"][i, j] = rp.smECE(y_pred, y_true)
            scores["brier_score"][i, j] = np.mean((y_true - y_pred) ** 2)
            scores["brier_score_cal"][i, j] = cal
            scores["brier_score_res"][i, j] = res
            scores["brier_score_unc"][i, j] = unc
            # scores["log_loss"][i, j] = -np.mean(y_true * np.log(y_pred + 1e-15) + (1 - y_true) * np.log(1 - y_pred + 1e-15))

    # plot
    x = np.arange(len(method_names), dtype=float)

    fig1, ax = Plotter.create()
    if single_plot:
        Plotter.set(ax, xticks=[])
    else:
        Plotter.set(ax, xticks=dict(ticks=x, labels=translate_method(method_names), rotation=label_rotation, ha="right"))
    Plotter.set(
        ax,
        ylim=YLIM_CALIBRATION,
    )
    plot_grouped_bar([
        (ax, scores["ece"], dict(label="ECE", color=COLOR_ECE, fmt="{:.2f}")),
        # (ax, scores["smECE"], dict(label="smECE", color=COLOR_SMECE, fmt="{:.2f}")),
        # (ax, scores["brier_score"], dict(label="Brier score", color=COLOR_BRIER, fmt="{:.2f}", alpha=0.0)),
        (ax, scores["brier_score"], dict(color=COLOR_BRIER, fmt="{:.2f}", alpha=0.0)),
    ], with_labels=True, with_lines=True, with_errorbars=True, none_indices=none_indices)
    plot_grouped_bar2(1, 2, [
        (ax, scores["brier_score_cal"], dict(label="Brier score (calibration)", color=COLOR_SMECE, fmt="\n\n{:.2f}" if hack else "{:.2f}")),
        (ax, -scores["brier_score_res"], dict(label="Brier score (resolution)", color=COLOR_CONF_VARIANCE, fmt="{:.2f}\n" if hack else "{:.2f}")),
        (ax, scores["brier_score_unc"], dict(label="Brier score (uncertainty)", color="tab:gray", fmt="{:.2f}\n" if hack else "{:.2f}")),
    ], with_labels=True, with_lines=True, with_errorbars=False, none_indices=none_indices)

    annotate_agg_over(ax, agg_over)

    return fig1

def plot(y_true_all, y_pred_all):
    dataset_names_all = [
        "sciq",
        "arc-e",
        "arc-c",
        "commonsense_qa",
        "social_i_qa",
        "mmlu",
        "trivia_qa",
        "truthful_qa-mc1",
        "truthful_qa-mc2",
        "logi_qa",
    ]
    model_names_all = [
        "gemma1.1-2b-it",
        "gemma1.1-7b-it",
        None,
        "llama3-8b-instruct",
        "llama3-70b-instruct",
        None,
        "qwen1.5-7b-chat",
        "qwen1.5-32b-chat",
        "qwen1.5-72b-chat",
        "qwen1.5-110b-chat",
        None,
        "gpt3.5-turbo",
        "gpt4o-mini",
        "gpt4o",
    ]

    plots = []

    fig1 = make_plots_datasets2(y_true_all, y_pred_all, "models[all], methods[all]", dataset_names=dataset_names_all, label_rotation=35)
    plots.append((fig1, "datasets-calibration_comparison"))

    fig1 = make_plots_models2(y_true_all, y_pred_all, "datasets[all], methods[all]", model_names=model_names_all)
    fig1.axes[0].legend(loc="center right")
    plots.append((fig1, "models-calibration_comparison"))

    fig1 = make_plots_methods2(y_true_all, y_pred_all, "datasets[all], models[tiny]", model_names=MODELS_TINY, method_names=METHODS_COMBO, label_rotation=25)
    plots.append((fig1, "methods-calibration_comparison-combo-tinymodels"))

    fig1 = make_plots_methods2(y_true_all, y_pred_all, "datasets[all], models[large]", model_names=MODELS_LARGE, method_names=METHODS_COMBO, label_rotation=25, hack=True)
    fig1.axes[0].legend(loc="center right")
    plots.append((fig1, "methods-calibration_comparison-combo-largemodels"))

    Plotter.finish(
        plots, figwidth=0.5,
        grid_ncols=2, consistent_size=True,
        # save=True,
    )

plot(y_true_all, y_pred_all)

In [ ]:
Plotter.configure(save_always=False)
plt.close()